# 01 Inventory

What is actually in the collection: tables, columns, inferred dtypes, null shares. Read
before any analysis, because every later claim is a claim about *these* files.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [2]:
from swsn.registry import load_registry
from swsn.tables import load_tables
from swsn.schema_doc import load_all_schemas

registry = load_registry("../data/datasets.yaml")
schemas = load_all_schemas("../ground_truth/schema")

rows = []
for did, schema in schemas.items():
    tables = load_tables(did, list(schema["tables"]), data_dir="../data", nrows=50_000)
    for name, df in tables.items():
        rows.append({
            "dataset": did,
            "table": name,
            "columns": len(df.columns),
            "rows (sampled)": len(df),
            "documented columns": len(schema["tables"][name].get("columns") or []),
            "null share": round(float(df.isna().mean().mean()), 4),
        })
pd.DataFrame(rows).sort_values(["dataset", "table"])

,dataset,table,columns,rows (sampled),documented columns,null share
4,instacart,aisles,2,134,2,0.0000
5,instacart,departments,2,21,2,0.0000
1,instacart,order_products__prior,4,50000,4,0.0000
2,instacart,order_products__train,4,50000,4,0.0000
0,instacart,orders,7,50000,7,0.0089
3,instacart,products,4,49688,4,0.0000
6,olist,olist_customers_dataset,5,50000,5,0.0000
7,olist,olist_geolocation_dataset,5,50000,5,0.0000
9,olist,olist_order_items_dataset,7,50000,7,0.0000
10,olist,olist_order_payments_dataset,5,50000,5,0.0000


## Column types, as pandas infers them

In [3]:
for did, schema in schemas.items():
    print(f"=== {did}")
    tables = load_tables(did, list(schema["tables"]), data_dir="../data", nrows=5_000)
    for name, df in tables.items():
        print(f"  {name}")
        print("   ", dict(df.dtypes.astype(str)))

=== instacart
  orders
    {'order_id': 'int64', 'user_id': 'int64', 'eval_set': 'str', 'order_number': 'int64', 'order_dow': 'int64', 'order_hour_of_day': 'int64', 'days_since_prior_order': 'float64'}
  order_products__prior
    {'order_id': 'int64', 'product_id': 'int64', 'add_to_cart_order': 'int64', 'reordered': 'int64'}
  order_products__train
    {'order_id': 'int64', 'product_id': 'int64', 'add_to_cart_order': 'int64', 'reordered': 'int64'}
  products
    {'product_id': 'int64', 'product_name': 'str', 'aisle_id': 'int64', 'department_id': 'int64'}
  aisles
    {'aisle_id': 'int64', 'aisle': 'str'}
  departments
    {'department_id': 'int64', 'department': 'str'}
=== olist
  olist_customers_dataset
    {'customer_id': 'str', 'customer_unique_id': 'str', 'customer_zip_code_prefix': 'int64', 'customer_city': 'str', 'customer_state': 'str'}
  olist_geolocation_dataset
    {'geolocation_zip_code_prefix': 'int64', 'geolocation_lat': 'float64', 'geolocation_lng': 'float64', 'geolocatio

Note the misspelling carried in the Olist source itself: `product_name_lenght` and
`product_description_lenght`. It is in the published schema too, so it is transcribed as
is — a canonicaliser that normalises Portuguese and English does not fix a typo.